In [1]:
!pip install requests pandas transformers torch

In [2]:
import requests
import pandas as pd
from transformers import pipeline

API_KEY = ""
CRYPTO_PANIC_URL = "https://cryptopanic.com/api/v1/posts/"


In [4]:
params = {
    "auth_token": API_KEY,
    "public": "true",
    "kind": "news"
}

response = requests.get(CRYPTO_PANIC_URL, params=params)
data = response.json()

news_list = []


In [7]:
data['results']

[{'id': 27924326,
  'slug': 'Ethereums-2026-Setup-Draws-Attention-as-Key-On-Chain-Metrics-Align',
  'title': 'Ethereum’s 2026 Setup Draws Attention as Key On-Chain Metrics Align',
  'description': 'Ethereum is back in focus after two influential analysts highlighted patterns that mirror previous cycle expansions. A combination of long-term technical structure and whale-level cost basis signals has revived speculation that ETH could be preparing for a major breakout heading into 2026.',
  'published_at': '2025-12-13T18:40:16Z',
  'created_at': '2025-12-13T18:40:16+00:00',
  'kind': 'news'},
 {'id': 27924327,
  'slug': 'Ripples-Brad-Garlinghouse-Shares-Key-Institutional-Signal-Makes-Bold-XRP-Prediction-For-2026',
  'title': 'Ripple’s Brad Garlinghouse Shares Key Institutional Signal, Makes Bold XRP Prediction For 2026\u202c',
  'description': 'Ripple CEO Brad Garlinghouse has outlined what he considers one of the clearest signs that institutional capital is finally entering the crypto ma

In [15]:
for item in data["results"]:
    news_list.append({
        "title": item["title"],
        "description":item["description"],
        "published_at": item["published_at"]
    })

df = pd.DataFrame(news_list)

# Combine title + summary
df["text"] = df["title"] + ". " + df["description"].str.strip()
df = df[df["text"] != ""]

In [16]:
print(f"Fetched {len(df)} news articles")

Fetched 60 news articles


In [17]:
df

,title,description,published_at,text
0,Ethereum’s 2026 Setup Draws Attention as Key O...,Ethereum is back in focus after two influentia...,2025-12-13T18:40:16Z,Ethereum’s 2026 Setup Draws Attention as Key O...
1,Ripple’s Brad Garlinghouse Shares Key Institut...,Ripple CEO Brad Garlinghouse has outlined what...,2025-12-13T18:37:27Z,Ripple’s Brad Garlinghouse Shares Key Institut...
2,Exor Rejects Tether's Acquisition Proposal for...,Exor rebuffs Tether's offer to acquire its Juv...,2025-12-13T18:34:39Z,Exor Rejects Tether's Acquisition Proposal for...
3,Coinbase to Add Prediction Market Ahead of Maj...,Coinbase plans to increase its wide range of c...,2025-12-13T18:33:00Z,Coinbase to Add Prediction Market Ahead of Maj...
4,Best Crypto to Buy Now? Holiday Drops Go Live ...,Crypto investors have begun searching for top ...,2025-12-13T18:30:57Z,Best Crypto to Buy Now? Holiday Drops Go Live ...
5,Almost One-Third of Bitcoin (BTC) Is Held by B...,Bitcoin's ownership profile is changing. Corpo...,2025-12-13T18:18:01Z,Almost One-Third of Bitcoin (BTC) Is Held by B...
6,iAero Launches Token Sweeper & LIQ Rewards,iAero unveils Token Sweeper and a six-month ca...,2025-12-13T18:13:00Z,iAero Launches Token Sweeper & LIQ Rewards. iA...
7,Rare Market Exhaustion Indicator Suggests Bitc...,"Bitcoin may be approaching a pivotal moment, a...",2025-12-13T18:00:48Z,Rare Market Exhaustion Indicator Suggests Bitc...
8,Solana ETFs Attract Significant Inflows Amid M...,"According to Cointelegraph, Solana (SOL) excha...",2025-12-13T17:53:46Z,Solana ETFs Attract Significant Inflows Amid M...
9,Bank CEO Issues Fraudulent Loans to Friends an...,A US bank executive is facing federal fraud ch...,2025-12-13T17:48:43Z,Bank CEO Issues Fraudulent Loans to Friends an...


In [13]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


In [18]:
texts = df["text"].astype(str).tolist()
results = sentiment_pipeline(
    texts,
    truncation=True
)

df["sentiment"] = [r["label"].lower() for r in results]
df["confidence"] = [r["score"] for r in results]


In [19]:
df.to_csv("crypto_news_finbert_sentiment.csv", index=False)
print("Saved results to crypto_news_finbert_sentiment.csv")

Saved results to crypto_news_finbert_sentiment.csv


In [22]:
df.head()

,title,description,published_at,text,sentiment,confidence
0,Ethereum’s 2026 Setup Draws Attention as Key O...,Ethereum is back in focus after two influentia...,2025-12-13T18:40:16Z,Ethereum’s 2026 Setup Draws Attention as Key O...,positive,0.940718
1,Ripple’s Brad Garlinghouse Shares Key Institut...,Ripple CEO Brad Garlinghouse has outlined what...,2025-12-13T18:37:27Z,Ripple’s Brad Garlinghouse Shares Key Institut...,positive,0.844455
2,Exor Rejects Tether's Acquisition Proposal for...,Exor rebuffs Tether's offer to acquire its Juv...,2025-12-13T18:34:39Z,Exor Rejects Tether's Acquisition Proposal for...,negative,0.867390
3,Coinbase to Add Prediction Market Ahead of Maj...,Coinbase plans to increase its wide range of c...,2025-12-13T18:33:00Z,Coinbase to Add Prediction Market Ahead of Maj...,positive,0.861603
4,Best Crypto to Buy Now? Holiday Drops Go Live ...,Crypto investors have begun searching for top ...,2025-12-13T18:30:57Z,Best Crypto to Buy Now? Holiday Drops Go Live ...,neutral,0.598799


In [26]:
print(df[["title","description","published_at","text","sentiment","confidence"]].head())

                                               title  \
0  Ethereum’s 2026 Setup Draws Attention as Key O...   
1  Ripple’s Brad Garlinghouse Shares Key Institut...   
2  Exor Rejects Tether's Acquisition Proposal for...   
3  Coinbase to Add Prediction Market Ahead of Maj...   
4  Best Crypto to Buy Now? Holiday Drops Go Live ...   

                                         description          published_at  \
0  Ethereum is back in focus after two influentia...  2025-12-13T18:40:16Z   
1  Ripple CEO Brad Garlinghouse has outlined what...  2025-12-13T18:37:27Z   
2  Exor rebuffs Tether's offer to acquire its Juv...  2025-12-13T18:34:39Z   
3  Coinbase plans to increase its wide range of c...  2025-12-13T18:33:00Z   
4  Crypto investors have begun searching for top ...  2025-12-13T18:30:57Z   

                                                text sentiment  confidence  
0  Ethereum’s 2026 Setup Draws Attention as Key O...  positive    0.940718  
1  Ripple’s Brad Garlinghouse Shares Key